In [ ]:
# Choosing `k` for daily outlier removal

Compare the IQR rule with `k = 1.5`, `k = 2`, and `k = 3` for `ABVERKAUFTE_MENGE_KG`.

Outliers are computed per (`ARTIKEL_ID`, `MARKT_ID`) series from `data/interim/transactions_daily_agg`.

## Setup

Import the required libraries, locate the daily aggregate parquet files, and open a DuckDB connection for the remaining analysis.

In [ ]:
from pathlib import Path
import os

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import duckdb
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "data" / "interim" / "transactions_daily_agg").exists():
        ROOT = candidate
        break

DATA_DIR = ROOT / "data" / "interim" / "transactions_daily_agg"
DATA_GLOB = DATA_DIR / "transactions_year_*.parquet"
DEMAND_COL = "ABVERKAUFTE_MENGE_KG"
TOP_N = 10

files = sorted(DATA_DIR.glob("transactions_year_*.parquet"))
if not files:
    raise FileNotFoundError(f"No parquet files found in {DATA_DIR}")

def sql_literal(value):
    return "'" + str(value).replace("'", "''") + "'"

con = duckdb.connect()
con.execute("PRAGMA threads=8")

print(f"Reading {len(files)} yearly parquet files from {DATA_DIR}")

## Source view

Create a DuckDB view over the yearly daily-aggregate files and check which descriptive columns are available for the later tables and plots.

In [ ]:
con.execute(
    f"""
    CREATE OR REPLACE TEMP VIEW source AS
    SELECT *, CAST(DATE AS DATE) AS DATE_D
    FROM read_parquet({sql_literal(DATA_GLOB)})
    """
)

columns = [row[0] for row in con.execute("DESCRIBE SELECT * FROM source").fetchall()]
requested_cols = [
    "ARTIKEL_ID",
    "MARKT_ID",
    "VERKAUFSEINHEIT",
    "ARTIKEL_BEZ",
    "UMS_MENGE",
    "GRAMM_BON",
    "DATE",
    DEMAND_COL,
]
available_cols = [col for col in requested_cols if col in columns]
missing_cols = [col for col in requested_cols if col not in columns]

if missing_cols:
    print("Requested columns not present in the daily aggregate files:", missing_cols)

available_cols

## Overall effect of `k = 1.5`, `k = 2`, and `k = 3`

Compute per-series IQR bounds and summarize how many rows would be removed globally for each candidate multiplier.  Percentage of removed rows across all series can be underrepresentative, since some series may experience stronger outlier filtering than other ones. Furthermore, the series are of different length, so relatively small filtering of long series can dominate the result even if shorter series experienced relatively big filtering. For analysis per series, see cells below.

In [ ]:
con.execute(
    f"""
    CREATE OR REPLACE TEMP TABLE bounds AS
    WITH q AS (
        SELECT
            ARTIKEL_ID,
            MARKT_ID,
            quantile_cont({DEMAND_COL}, 0.25) AS q1,
            quantile_cont({DEMAND_COL}, 0.75) AS q3
        FROM source
        GROUP BY ARTIKEL_ID, MARKT_ID
    )
    SELECT
        ARTIKEL_ID,
        MARKT_ID,
        q1,
        q3,
        q3 - q1 AS iqr,
        q3 + 1.5 * (q3 - q1) AS upper_k15,
        q3 + 2 * (q3 - q1) AS upper_k2,
        q3 + 3 * (q3 - q1) AS upper_k3
    FROM q
    """
)

summary = con.execute(
    f"""
    SELECT
        COUNT(*) AS rows_total,
        SUM(CASE WHEN s.{DEMAND_COL} > b.upper_k15 THEN 1 ELSE 0 END) AS rows_removed_k15,
        SUM(CASE WHEN s.{DEMAND_COL} > b.upper_k2 THEN 1 ELSE 0 END) AS rows_removed_k2,
        SUM(CASE WHEN s.{DEMAND_COL} > b.upper_k3 THEN 1 ELSE 0 END) AS rows_removed_k3
    FROM source s
    JOIN bounds b USING (ARTIKEL_ID, MARKT_ID)
    """
).df()

summary["pct_removed_k15"] = summary["rows_removed_k15"] / summary["rows_total"] * 100
summary["pct_removed_k2"] = summary["rows_removed_k2"] / summary["rows_total"] * 100
summary["pct_removed_k3"] = summary["rows_removed_k3"] / summary["rows_total"] * 100
summary

## CV before and after outlier removal

Compute the coefficient of variation (`stddev_samp / mean`) per product-store series before filtering and after applying each candidate `k`, then compare the resulting CV distributions.

In [ ]:
cv_by_k = con.execute(
    f"""
    SELECT
        s.ARTIKEL_ID,
        s.MARKT_ID,
        stddev_samp(s.{DEMAND_COL}) / NULLIF(avg(s.{DEMAND_COL}), 0) AS cv_before,
        stddev_samp(CASE WHEN s.{DEMAND_COL} <= b.upper_k15 THEN s.{DEMAND_COL} END)
            / NULLIF(avg(CASE WHEN s.{DEMAND_COL} <= b.upper_k15 THEN s.{DEMAND_COL} END), 0) AS cv_after_k15,
        stddev_samp(CASE WHEN s.{DEMAND_COL} <= b.upper_k2 THEN s.{DEMAND_COL} END)
            / NULLIF(avg(CASE WHEN s.{DEMAND_COL} <= b.upper_k2 THEN s.{DEMAND_COL} END), 0) AS cv_after_k2,
        stddev_samp(CASE WHEN s.{DEMAND_COL} <= b.upper_k3 THEN s.{DEMAND_COL} END)
            / NULLIF(avg(CASE WHEN s.{DEMAND_COL} <= b.upper_k3 THEN s.{DEMAND_COL} END), 0) AS cv_after_k3
    FROM source s
    JOIN bounds b USING (ARTIKEL_ID, MARKT_ID)
    GROUP BY s.ARTIKEL_ID, s.MARKT_ID
    """
).df()

cv_plot_cols = {
    "vorher": "cv_before",
    "nach k = 1,5": "cv_after_k15",
    "nach k = 2": "cv_after_k2",
    "nach k = 3": "cv_after_k3",
}
cv_boxplot_data = [cv_by_k[col].dropna() for col in cv_plot_cols.values()]

fig, ax = plt.subplots(figsize=(9, 5))
box = ax.boxplot(
    cv_boxplot_data,
    tick_labels=list(cv_plot_cols.keys()),
    showfliers=False,
    patch_artist=True,
)
for patch, color in zip(box["boxes"], ["0.6", "tab:red", "tab:orange", "tab:blue"]):
    patch.set_facecolor(color)
    patch.set_alpha(0.35)

ax.set_title("CV-Verteilung vor und nach Ausreißerbehandlung")
ax.set_xlabel("Filter-Szenario")
ax.set_ylabel("Variationskoeffizient")
ax.set_ylim(bottom=0)
ax.grid(True, axis="y", alpha=0.25)

plt.tight_layout()
plt.show()

## Median CV and row-removal effect by `k`

Summarize the filtering effect in one table using one shared population per `k`: only series with valid positive `cv_before` and valid `cv_after_k` are used for the CV-reduction median, the median percentage of rows removed, and the percentage of affected series.

The coefficient of variation is calculated as `CV = stddev_samp(ABVERKAUFTE_MENGE_KG) / mean(ABVERKAUFTE_MENGE_KG)`. The reduction is calculated per series as `100 * (cv_before - cv_after_k) / cv_before`.

`median rows removed, %` is also calculated per series first as `100 * rows_removed_in_series / rows_total_in_series`, then summarized with the median across valid series. This avoids longer series dominating the comparison.

`% affected series` is the share of valid series with at least one row removed for the given `k`. Series with `CV = 0` are excluded from all three summary columns because they have no standard deviation, so there is no variation-driven outlier to remove. If the mean were `0` while the standard deviation were positive, CV would be undefined rather than `0`; this is not the case here because the transaction quantities used in this notebook are positive.

In [ ]:
per_series_removed = con.execute(
    f"""
    SELECT
        s.ARTIKEL_ID,
        s.MARKT_ID,
        COUNT(*) AS rows_total,
        SUM(CASE WHEN s.{DEMAND_COL} > b.upper_k15 THEN 1 ELSE 0 END) AS rows_removed_k15,
        SUM(CASE WHEN s.{DEMAND_COL} > b.upper_k2 THEN 1 ELSE 0 END) AS rows_removed_k2,
        SUM(CASE WHEN s.{DEMAND_COL} > b.upper_k3 THEN 1 ELSE 0 END) AS rows_removed_k3,
        100.0 * SUM(CASE WHEN s.{DEMAND_COL} > b.upper_k15 THEN 1 ELSE 0 END) / COUNT(*) AS pct_removed_k15,
        100.0 * SUM(CASE WHEN s.{DEMAND_COL} > b.upper_k2 THEN 1 ELSE 0 END) / COUNT(*) AS pct_removed_k2,
        100.0 * SUM(CASE WHEN s.{DEMAND_COL} > b.upper_k3 THEN 1 ELSE 0 END) / COUNT(*) AS pct_removed_k3
    FROM source s
    JOIN bounds b USING (ARTIKEL_ID, MARKT_ID)
    GROUP BY s.ARTIKEL_ID, s.MARKT_ID
    """
).df()

cv_effect = cv_by_k.merge(
    per_series_removed[
        [
            "ARTIKEL_ID",
            "MARKT_ID",
            "pct_removed_k15",
            "pct_removed_k2",
            "pct_removed_k3",
        ]
    ],
    on=["ARTIKEL_ID", "MARKT_ID"],
    how="inner",
)

k_summary_specs = [
    (1.5, "cv_after_k15", "pct_removed_k15"),
    (2.0, "cv_after_k2", "pct_removed_k2"),
    (3.0, "cv_after_k3", "pct_removed_k3"),
]

summary_rows = []
for k, cv_col, rows_col in k_summary_specs:
    valid_k = cv_effect.loc[
        cv_effect["cv_before"].notna()
        & (cv_effect["cv_before"] > 0)
        & cv_effect[cv_col].notna()
    ].copy()
    valid_k["cv_reduction_pct"] = (
        100.0 * (valid_k["cv_before"] - valid_k[cv_col]) / valid_k["cv_before"]
    )
    summary_rows.append(
        {
            "k": k,
            "median cv reduction, %": valid_k["cv_reduction_pct"].median(),
            "median rows removed, %": valid_k[rows_col].median(),
            "affected series, %": 100.0 * (valid_k[rows_col] > 0).mean(),
        }
    )

cv_reduction_by_k = pd.DataFrame(summary_rows)

cv_reduction_by_k = cv_reduction_by_k.round(
    {
        "median cv reduction, %": 2,
        "median rows removed, %": 2,
        "affected series, %": 2,
    }
)
cv_reduction_by_k

## Ten product/store series with the biggest outlier rows

The table keeps one biggest outlier row per (`ARTIKEL_ID`, `MARKT_ID`) series, ranked by the excess over the `k = 1.5` upper bound.

In [ ]:
select_cols = ",\n            ".join(f"s.{col}" for col in available_cols)

top_outliers = con.execute(
    f"""
    WITH outlier_rows AS (
        SELECT
            {select_cols},
            b.q1,
            b.q3,
            b.iqr,
            b.upper_k15,
            b.upper_k2,
            b.upper_k3,
            s.{DEMAND_COL} - b.upper_k15 AS excess_k15,
            s.{DEMAND_COL} - b.upper_k2 AS excess_k2,
            s.{DEMAND_COL} - b.upper_k3 AS excess_k3,
            s.{DEMAND_COL} > b.upper_k15 AS outlier_k15,
            s.{DEMAND_COL} > b.upper_k2 AS outlier_k2,
            s.{DEMAND_COL} > b.upper_k3 AS outlier_k3,
            ROW_NUMBER() OVER (
                PARTITION BY s.ARTIKEL_ID, s.MARKT_ID
                ORDER BY s.{DEMAND_COL} - b.upper_k15 DESC
            ) AS series_rank
        FROM source s
        JOIN bounds b USING (ARTIKEL_ID, MARKT_ID)
        WHERE s.{DEMAND_COL} > b.upper_k15
    )
    SELECT * EXCLUDE (series_rank)
    FROM outlier_rows
    WHERE series_rank = 1
    ORDER BY excess_k15 DESC
    LIMIT {TOP_N}
    """
).df()

top_outliers

## Raw series vs after removing outliers

Plot the highest-impact product-store series with each candidate cutoff.

In [ ]:
if top_outliers.empty:
    raise ValueError("No outliers found with k = 1.5.")

top_series = top_outliers[["ARTIKEL_ID", "MARKT_ID"]].copy()
con.register("top_series", top_series)

plot_df = con.execute(
    f"""
    SELECT
        s.ARTIKEL_ID,
        s.MARKT_ID,
        s.DATE_D AS DATE,
        s.{DEMAND_COL},
        s.ARTIKEL_BEZ,
        s.VERKAUFSEINHEIT,
        b.upper_k15,
        b.upper_k2,
        b.upper_k3
    FROM source s
    JOIN top_series ts USING (ARTIKEL_ID, MARKT_ID)
    JOIN bounds b USING (ARTIKEL_ID, MARKT_ID)
    ORDER BY s.ARTIKEL_ID, s.MARKT_ID, s.DATE_D
    """
).df()

plot_df["DATE"] = pd.to_datetime(plot_df["DATE"])
plot_df["after_k15"] = plot_df[DEMAND_COL].where(plot_df[DEMAND_COL] <= plot_df["upper_k15"])
plot_df["after_k2"] = plot_df[DEMAND_COL].where(plot_df[DEMAND_COL] <= plot_df["upper_k2"])
plot_df["after_k3"] = plot_df[DEMAND_COL].where(plot_df[DEMAND_COL] <= plot_df["upper_k3"])

groups = list(plot_df.groupby(["ARTIKEL_ID", "MARKT_ID"], sort=False))
store_labels = {
    key: f"Filiale {chr(ord('A') + idx)}"
    for idx, (key, _) in enumerate(groups)
}

example_series_lookup = pd.DataFrame(
    [
        {
            "store_label": store_labels[(artikel_id, markt_id)],
            "ARTIKEL_ID": artikel_id,
            "MARKT_ID": markt_id,
            "ARTIKEL_BEZ": str(df["ARTIKEL_BEZ"].iloc[0]),
            "VERKAUFSEINHEIT": df["VERKAUFSEINHEIT"].iloc[0],
        }
        for (artikel_id, markt_id), df in groups
    ]
)
display(example_series_lookup)

fig, axes = plt.subplots(len(groups), 1, figsize=(14, 3.2 * len(groups)), sharex=False)
if len(groups) == 1:
    axes = [axes]

for ax, ((artikel_id, markt_id), df) in zip(axes, groups):
    title = str(df["ARTIKEL_BEZ"].iloc[0])[:90]
    store_label = store_labels[(artikel_id, markt_id)]
    unit = df["VERKAUFSEINHEIT"].iloc[0]

    ax.plot(df["DATE"], df[DEMAND_COL], color="0.35", linewidth=1.0, label="Original")
    ax.plot(df["DATE"], df["after_k15"], color="tab:red", linewidth=1.1, label="nach k=1,5")
    ax.plot(df["DATE"], df["after_k2"], color="tab:orange", linewidth=1.1, label="nach k=2")
    ax.plot(df["DATE"], df["after_k3"], color="tab:blue", linewidth=1.1, label="nach k=3")
    ax.axhline(df["upper_k15"].iloc[0], color="tab:red", linestyle="--", linewidth=0.8, alpha=0.7)
    ax.axhline(df["upper_k2"].iloc[0], color="tab:orange", linestyle="--", linewidth=0.8, alpha=0.7)
    ax.axhline(df["upper_k3"].iloc[0], color="tab:blue", linestyle="--", linewidth=0.8, alpha=0.7)

    ax.set_title(f"{title} - {store_label}, Einheit={unit}")
    ax.set_ylabel("Absatzmenge")
    ax.legend(loc="upper right")

plt.tight_layout()